# Starter notebook: E. coli, Hospital A (Members 1 and 2)

**What this gives you:** the shared data loader and `run_cv`, the common test every method must use, so all results can be compared.

**Setup (once):** put `spectra_ecoli_A.npy` and `labels_ecoli_A.csv` in a folder named `shared_data` next to this notebook.

**Rules**
- Do not change `run_cv`, the folds or `MODEL_ABX`.
- Save each method as a CSV with exactly these columns: `antibiotic, method, auc, auc_std, prauc, n_features`.
- Give each method its own `method` name.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Spectra: one row per E. coli isolate, 6,000 bins (columns)
X = np.load("shared_data/spectra_ecoli_A.npy")

# Labels: R=1, S=0, empty = not tested. fold_<antibiotic> = fold number 0-4 (-1 = no label)
lab = pd.read_csv("shared_data/labels_ecoli_A.csv")

# The 12 antibiotics we model (the other 3 have too few R or S)
MODEL_ABX = ["Ampicillin-Amoxicillin", "Cotrimoxazole", "Ciprofloxacin", "Levofloxacin",
             "Amoxicillin-Clavulanic acid", "Ceftriaxone", "Cefpodoxime", "Tobramycin",
             "Ceftazidime", "Norfloxacin", "Cefepime", "Piperacillin-Tazobactam"]

print(X.shape, lab.shape)   # expected (7320, 6000) (7320, 32)

## `run_cv`: the common evaluation function

For one antibiotic: use the saved folds (train on 4, test on 1, repeat 5 times).
The reducer is fitted on the **training part only** (no leakage), then Logistic Regression is trained.
It returns AUC (Area Under the ROC Curve), PR-AUC (Precision-Recall AUC) and the number of features used.

In [ ]:
def run_cv(reducer, antibiotic):
    """reducer = object with fit(X, y) / transform(X), or None for no reduction."""
    has_label = (lab[antibiotic].notna()) & (lab[f"fold_{antibiotic}"] >= 0)
    Xa = X[has_label.values]                                   # spectra with an R/S result
    ya = lab.loc[has_label, antibiotic].astype(int).values     # R=1, S=0
    fa = lab.loc[has_label, f"fold_{antibiotic}"].values       # fold number 0-4

    aucs, praucs, n_feats = [], [], []
    for k in range(5):
        tr, te = fa != k, fa == k                  # train = other 4 folds, test = fold k
        steps = [StandardScaler()]                 # put all bins on the same scale
        if reducer is not None:
            steps.append(clone(reducer))           # fresh copy, fitted on train only
        steps.append(LogisticRegression(max_iter=1000))   # simple classifier
        model = make_pipeline(*steps).fit(Xa[tr], ya[tr])

        p = model.predict_proba(Xa[te])[:, 1]      # probability of resistant
        aucs.append(roc_auc_score(ya[te], p))
        praucs.append(average_precision_score(ya[te], p))
        n_feats.append(model[:-1].transform(Xa[te][:5]).shape[1])   # features the classifier saw

    return {"antibiotic": antibiotic, "auc": np.mean(aucs), "auc_std": np.std(aucs),
            "prauc": np.mean(praucs), "n_features": int(np.mean(n_feats))}

## Quick test (any reducer works the same way)

Compare with our Cefepime baseline: AUC 0.840, PR-AUC 0.403 (all 6,000 bins).

In [ ]:
from sklearn.decomposition import PCA   # PCA = Principal Component Analysis

# Replace PCA(...) with your own method (Kernel PCA, PSO selector, ...)
print(run_cv(PCA(n_components=50), "Cefepime"))

## Template for your own method

A reducer is a class with `fit(X, y)` and `transform(X)`.
`LassoSelect` below is a working example (it keeps the bins with non-zero LASSO weight). Copy its structure for PSO or any other selector.

In [ ]:
class LassoSelect(BaseEstimator, TransformerMixin):
    def __init__(self, C=0.03):          # C = penalty strength (small C = fewer bins kept)
        self.C = C

    def fit(self, X, y):
        m = LogisticRegression(penalty="l1", solver="liblinear", C=self.C, max_iter=1000).fit(X, y)
        w = np.abs(m.coef_[0])           # importance of each bin
        self.mask_ = w > 0               # keep only the bins with non-zero weight
        if not self.mask_.any():         # safety: keep 1 bin if all were removed
            self.mask_[0] = True
        return self                      # fit must return self

    def transform(self, X):
        return X[:, self.mask_]          # return only the kept bins (a PSO selector would return its chosen bins)

print(run_cv(LassoSelect(0.03), "Cefepime"))   # our LASSO result: AUC 0.871

## Run all 12 antibiotics and save the results

In [ ]:
METHOD_NAME = "lasso_C0.03"    # change to your method name
my_reducer = LassoSelect(0.03) # change to your method

rows = []
for abx in MODEL_ABX:
    r = run_cv(my_reducer, abx)
    r["method"] = METHOD_NAME
    rows.append(r)
    print(abx, "AUC =", round(r["auc"], 3), "| PR-AUC =", round(r["prauc"], 3), "| features =", r["n_features"])

res = pd.DataFrame(rows)[["antibiotic", "method", "auc", "auc_std", "prauc", "n_features"]]
res.to_csv(f"results_{METHOD_NAME}.csv", index=False)   # send this file back
display(res)

## Merge (done later, when all result files are in one folder)

In [ ]:
import glob
all_results = pd.concat([pd.read_csv(f) for f in glob.glob("results_*.csv")])
# Rows = antibiotics, columns = methods, values = AUC
display(all_results.pivot(index="antibiotic", columns="method", values="auc").round(3))